In [ ]:
# ==============================================================================
# CELL 1: Setup & Download Dataset Subhan
# ==============================================================================
!pip install -q kagglehub xgboost imbalanced-learn opencv-python scikit-image

import os
import cv2
import numpy as np
import joblib
import warnings
import kagglehub
import random
from collections import Counter

from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from sklearn.svm import SVC
from tqdm.notebook import tqdm
from google.colab import files

warnings.filterwarnings('ignore')
os.environ["KAGGLE_API_TOKEN"] = "KGAT_a3d02acd0f42e7804e0119e00820db3f"

print("--- MENGUNDUH DATASET SUBHAN ---")
path_disease = kagglehub.dataset_download("muhammad0subhan/fruit-and-vegetable-disease-healthy-vs-rotten")
PATH_DISEASE_ROOT = None
for root, dirs, _ in os.walk(path_disease):
    if any("__" in d for d in dirs):
        PATH_DISEASE_ROOT = root
        break

TARGET_CLASSES = ['Apple', 'Banana', 'Grape', 'Guava', 'Mango', 'Orange', 'Pomegranate', 'Strawberry', 'Tomato']

print(f"Dataset: {PATH_DISEASE_ROOT}")
print(f"Target : {TARGET_CLASSES}")

# Cek folder yang tersedia
available = [d for d in os.listdir(PATH_DISEASE_ROOT) if "__" in d]
print(f"\nFolder tersedia ({len(available)}): {sorted(available)}")


In [ ]:
# ==============================================================================
# CELL 2: Fungsi Ekstraksi Fitur (Enhanced v3)
# ==============================================================================
IMG_SIZE = 128
RADIUS = 3
N_POINTS = 8 * RADIUS

def extract_advanced_features_segmented(img_bgr):
    if img_bgr is None: return None
    img_resized = cv2.resize(img_bgr, (IMG_SIZE, IMG_SIZE))

    gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(img_resized, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(img_resized, cv2.COLOR_BGR2LAB)
    h, s, v = cv2.split(hsv)

    # Segmentasi: GrabCut + fallback Otsu
    mask_grabcut = np.zeros(img_resized.shape[:2], np.uint8)
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)
    margin = IMG_SIZE // 8
    rect = (margin, margin, IMG_SIZE - 2 * margin, IMG_SIZE - 2 * margin)
    try:
        cv2.grabCut(img_resized, mask_grabcut, rect, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_RECT)
        mask = np.where((mask_grabcut == 2) | (mask_grabcut == 0), 0, 255).astype(np.uint8)
        if mask.sum() < (IMG_SIZE * IMG_SIZE * 0.05 * 255):
            raise ValueError("mask terlalu kecil")
    except:
        _, thresh_s = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        _, thresh_gray = cv2.threshold(gray, 220, 255, cv2.THRESH_BINARY_INV)
        mask = cv2.bitwise_or(thresh_s, thresh_gray)

    kernel = np.ones((9, 9), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        mask.fill(255)
        final_contour = np.array([[[0,0]], [[0,IMG_SIZE]], [[IMG_SIZE,IMG_SIZE]], [[IMG_SIZE,0]]])
    else:
        largest_contour = max(contours, key=cv2.contourArea)
        final_contour = cv2.convexHull(largest_contour)
        mask = np.zeros_like(gray)
        cv2.drawContours(mask, [final_contour], -1, 255, thickness=cv2.FILLED)

    # Color: histogram 32 bins
    hist_h = cv2.calcHist([hsv], [0], mask, [32], [0, 256]).flatten()
    hist_s = cv2.calcHist([hsv], [1], mask, [32], [0, 256]).flatten()
    hist_a = cv2.calcHist([lab], [1], mask, [32], [0, 256]).flatten()
    hist_b = cv2.calcHist([lab], [2], mask, [32], [0, 256]).flatten()
    color_hist = np.concatenate([hist_h, hist_s, hist_a, hist_b])
    color_hist /= (color_hist.sum() + 1e-7)

    # Color: mean, std, Q25, Q75 per channel
    channels = [h, s, v,
                lab[:,:,0], lab[:,:,1], lab[:,:,2],
                img_resized[:,:,0], img_resized[:,:,1], img_resized[:,:,2]]
    color_stats = []
    for ch in channels:
        masked_vals = ch[mask > 0].astype(np.float32)
        if len(masked_vals) == 0:
            color_stats.extend([0, 0, 0, 0])
        else:
            color_stats.extend([
                masked_vals.mean(),
                masked_vals.std(),
                float(np.percentile(masked_vals, 25)),
                float(np.percentile(masked_vals, 75))
            ])
    color_stats = np.array(color_stats) / 255.0

    # Dominant color k-means k=3
    pixels = img_resized[mask > 0].reshape(-1, 3).astype(np.float32)
    dominant_color_feats = np.zeros(9)
    if len(pixels) >= 3:
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
        _, labels, centers = cv2.kmeans(pixels, 3, None, criteria, 3, cv2.KMEANS_RANDOM_CENTERS)
        counts = np.bincount(labels.flatten(), minlength=3).astype(np.float32)
        order = np.argsort(-counts)
        dominant_color_feats = (centers[order] / 255.0).flatten()

    # Texture: GLCM
    gray_masked = cv2.bitwise_and(gray, gray, mask=mask)
    angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    distances = [1, 3, 5]
    glcm = graycomatrix(gray_masked, distances=distances, angles=angles, levels=256, symmetric=True, normed=True)
    glcm_features = np.concatenate([
        graycoprops(glcm, 'contrast').flatten(),
        graycoprops(glcm, 'correlation').flatten(),
        graycoprops(glcm, 'energy').flatten(),
        graycoprops(glcm, 'homogeneity').flatten()
    ])

    # Texture: LBP
    lbp = local_binary_pattern(gray_masked, N_POINTS, RADIUS, method='uniform')
    hist_lbp, _ = np.histogram(lbp.ravel(), bins=np.arange(0, N_POINTS + 3), range=(0, N_POINTS + 2))
    hist_lbp = hist_lbp.astype(float)
    hist_lbp /= (hist_lbp.sum() + 1e-7)

    # Shape: Hu Moments + geometric
    moments_c = cv2.moments(final_contour)
    hu_moments = cv2.HuMoments(moments_c).flatten()
    hu_moments = -np.sign(hu_moments) * np.log10(np.abs(hu_moments) + 1e-10)

    area = cv2.contourArea(final_contour)
    perimeter = cv2.arcLength(final_contour, True)
    circularity = (4 * np.pi * area / (perimeter ** 2 + 1e-7))
    x, y_bb, w, h_bbox = cv2.boundingRect(final_contour)
    aspect_ratio = w / (h_bbox + 1e-7)
    extent = area / (w * h_bbox + 1e-7)
    hull_area = cv2.contourArea(cv2.convexHull(final_contour))
    solidity = area / (hull_area + 1e-7)
    equiv_diameter = np.sqrt(4 * area / np.pi + 1e-7)
    shape_features = np.array([circularity, aspect_ratio, extent, solidity, equiv_diameter / IMG_SIZE])

    # Shape: distribusi jarak kontur ke centroid
    if moments_c["m00"] != 0:
        cx = int(moments_c["m10"] / moments_c["m00"])
        cy = int(moments_c["m01"] / moments_c["m00"])
    else:
        cx, cy = IMG_SIZE // 2, IMG_SIZE // 2

    if len(final_contour) > 0:
        pts = final_contour.reshape(-1, 2).astype(np.float32)
        dists = np.sqrt((pts[:, 0] - cx)**2 + (pts[:, 1] - cy)**2)
        dists_norm = dists / (dists.max() + 1e-7)
        shape_hist, _ = np.histogram(dists_norm, bins=8, range=(0, 1))
        shape_hist = shape_hist.astype(float)
        shape_hist /= (shape_hist.sum() + 1e-7)
    else:
        shape_hist = np.zeros(8)

    return np.concatenate([
        color_hist, color_stats, dominant_color_feats,
        glcm_features, hist_lbp, hu_moments,
        shape_features, shape_hist
    ])


In [ ]:
# ==============================================================================
# CELL 3: Ekstraksi Fitur Semua Buah
# ==============================================================================
print("Mengekstrak fitur dari dataset Subhan...\n")

all_features = {}  # { 'Apple': {'X': [...], 'y': [...]} }

for target in TARGET_CLASSES:
    X_raw, y_raw = [], []

    for folder_name in os.listdir(PATH_DISEASE_ROOT):
        if "__" not in folder_name: continue
        parts = folder_name.split("__")
        if len(parts) != 2: continue
        fruit_name, condition = parts
        if fruit_name != target: continue

        folder_path = os.path.join(PATH_DISEASE_ROOT, folder_name)
        images = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        random.shuffle(images)

        for img_name in tqdm(images[:300], desc=f"{target}/{condition}", leave=False):
            img = cv2.imread(os.path.join(folder_path, img_name))
            feat = extract_advanced_features_segmented(img)
            if feat is not None:
                X_raw.append(feat)
                y_raw.append(condition)
                # Augmentasi: horizontal flip
                feat_flip = extract_advanced_features_segmented(cv2.flip(img, 1))
                if feat_flip is not None:
                    X_raw.append(feat_flip)
                    y_raw.append(condition)

    if len(X_raw) == 0:
        print(f"[SKIP] {target}: tidak ada data")
        continue

    all_features[target] = {
        'X': np.array(X_raw),
        'y': np.array(y_raw)
    }

    dist = Counter(y_raw)
    print(f"{target}: {dict(dist)} -> total {len(y_raw)}")

print("\n[SELESAI] Ekstraksi fitur selesai.")


In [ ]:
# ==============================================================================
# CELL 4: Training Spesialis Disease per Buah
# ==============================================================================
print("Melatih spesialis per buah...\n")

# Scaler global — fit dari semua data digabung
all_X = np.concatenate([all_features[t]['X'] for t in all_features])
global_scaler = StandardScaler()
global_scaler.fit(all_X)
print(f"Global scaler fit dari {len(all_X)} sampel total\n")

disease_models = {}
le_diseases = {}
disease_classes_per_fruit = {}

for target in TARGET_CLASSES:
    if target not in all_features:
        print(f"[SKIP] {target}: tidak ada data")
        continue

    print(f"[ {target} ]")
    X = all_features[target]['X']
    y = all_features[target]['y']

    if len(np.unique(y)) < 2:
        print(f"  -> Skip: hanya 1 kelas\n")
        continue

    X_scaled = global_scaler.transform(X)
    le = LabelEncoder()
    y_enc = le.fit_transform(y)
    le_diseases[target] = le
    disease_classes_per_fruit[target] = list(le.classes_)

    print(f"  Kelas: {list(le.classes_)}")

    X_tr, X_vl, y_tr, y_vl = train_test_split(
        X_scaled, y_enc, test_size=0.20, random_state=42, stratify=y_enc
    )
    print(f"  Train: {len(X_tr)} | Val: {len(X_vl)}")

    sample_weights = compute_sample_weight(class_weight='balanced', y=y_tr)

    # XGBoost
    xgb = XGBClassifier(
        max_depth=3,
        learning_rate=0.03,
        n_estimators=150,
        reg_lambda=15.0,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.7,
        tree_method='hist',
        device='cuda',
        random_state=42
    )
    xgb.fit(X_tr, y_tr, sample_weight=sample_weights,
            eval_set=[(X_vl, y_vl)], verbose=False)
    acc_xgb = accuracy_score(y_vl, xgb.predict(X_vl))

    # SVM
    svm_params = {'C': [0.1, 1], 'gamma': ['scale']}
    gs_svm = GridSearchCV(SVC(probability=True, random_state=42), svm_params, cv=3, n_jobs=-1)
    gs_svm.fit(X_tr, y_tr)
    acc_svm = accuracy_score(y_vl, gs_svm.predict(X_vl))

    if acc_svm > acc_xgb:
        disease_models[target] = {'model': gs_svm.best_estimator_, 'type': 'svm'}
        print(f"  -> SVM terpilih (val acc: {acc_svm:.3f} vs XGB: {acc_xgb:.3f})\n")
    else:
        disease_models[target] = {'model': xgb, 'type': 'tree'}
        print(f"  -> XGBoost terpilih (val acc: {acc_xgb:.3f} vs SVM: {acc_svm:.3f})\n")

print("[SELESAI] Semua spesialis selesai dilatih.")


In [ ]:
# ==============================================================================
# CELL 5: Simpan Pipeline
# ==============================================================================
export_pipeline = {
    'architecture_type': 'Disease-Only Specialist v4 (Subhan Dataset)',
    'disease_models': disease_models,
    'global_scaler': global_scaler,
    'le_diseases': le_diseases,
    'target_classes': TARGET_CLASSES,
    'disease_classes': disease_classes_per_fruit
}

file_name = 'cascade_ultimate_hybrid_claude.pkl'
joblib.dump(export_pipeline, file_name)
print(f"[ SELESAI ] Pipeline disimpan ke '{file_name}'")
print(f"Buah tersedia: {list(disease_models.keys())}")
for fruit, info in disease_models.items():
    print(f"  {fruit}: {disease_classes_per_fruit[fruit]} ({info['type']})")

try:
    files.download(file_name)
except:
    print(f"Unduh manual dari panel kiri Colab: {file_name}")
